**Learning Curves for Linear Regression**

Learning curves help us understand how a model's performance changes as we increase the training data size. A plot of training error and validation error versus training set size can provide insights into whether a model is underfitting, overfitting, or performing well.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection  import train_test_split
from sklearn.metrics          import mean_squared_error
from sklearn.datasets         import fetch_openml
import plotly.express as px


from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline      import Pipeline
from sklearn.impute        import SimpleImputer


In [2]:
# Load the California housing dataset from OpenML
housing_sale = fetch_openml(name='house_sales', version=1,parser='pandas',target_column='price')
housing_sale.data.drop('date',axis=1,inplace=True)
X = housing_sale.data
y = housing_sale.target

/tmp/ipython-input-223570986.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  housing_sale.data.drop('date',axis=1,inplace=True)


In [3]:
# Display data types to check for categorical features
print(X.dtypes)

bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object


In [4]:
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
numeric_features     = X.select_dtypes(include=['int64','float64']).columns.tolist()

# Preprocessor for numeric data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessor for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing for both types of data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Preprocess the data
X_processed = preprocessor.fit_transform(X)




In [5]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

**A Function to plot the Learning Curves**

In [10]:
def plot_learning_curves_plotly(model, X_train, y_train, X_val, y_val, start_size=20, end_size=200):

    train_errors, val_errors = [], []

    for m in range(start_size, end_size):

        model.fit(X_train[:m], y_train[:m])  # Fit the model on the subset of the training data

        # Predict on the training and validation sets
        y_train_predict = model.predict(X_train[:m])     #Train data
        y_val_predict  = model.predict(X_val)             # Test Data

        # Calculate mean squared error for both sets
        train_errors.append(mean_squared_error(y_train[:m], y_train_predict)) #Train
        val_errors.append(mean_squared_error(y_val, y_val_predict)) # T

    # Convert errors to square root (Root Mean Squared Error)
    train_errors = np.sqrt(train_errors)
    val_errors   = np.sqrt(val_errors)

    # Prepare data for Plotly Express
    data = {
        "Training Set Size": list(range(start_size, end_size)),
        "Training Error"   : train_errors,
        "Validation|Test Error" : val_errors
    }

    # Plot the learning curves
    fig = px.line(
        data_frame=data,
        x="Training Set Size",
        y=["Training Error", "Validation|Test Error"],
        labels={"value": "Root Mean Squared Error (RMSE)", "variable": "Error Type"},
        title="Learning Curves"
    )
    fig.show()



In [11]:
#Apply To Linear Regression without regularization

from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
plot_learning_curves_plotly(lin_reg, X_train, y_train, X_test, y_test)

**Lets Traina a Linear Model Without Regularization**

In [12]:
lr_model  = LinearRegression()
lr_model.fit(X_train,y_train)
y_pred= lr_model.predict(X_test)

print("Ridge Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


Ridge Regression RMSE: 212539.51663817756


**Regularized Linear Models**

Ridge Regression - Applies L2 regularization to prevent overfitting.


$$
J(\beta) = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^p \beta_j
$$

---


- Ridge regression shrinks coefficients by penalizing large values, which makes the model more stable and less sensitive to fluctuations in the training data.

This reduces variance, improving generalization to unseen data.

The regularization parameter
𝛼- α controls the strength of the penalty:

Higher
𝛼 α → more shrinkage → simpler model → less overfitting.

Lower
𝛼 α → behaves more like standard linear regression.

In [27]:
from sklearn.linear_model import Ridge

# Ridge Regression
ridge_reg = Ridge(alpha=.1)
ridge_reg.fit(X_train, y_train)
y_pred_ridge = ridge_reg.predict(X_test)

print("Ridge Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))

Ridge Regression RMSE: 212539.57891823162


In [28]:
plot_learning_curves_plotly(ridge_reg, X_train, y_train, X_test, y_test)

Lasso Regression - Applies L1 regularization.

$$
J(\beta) = \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^p |\beta_j|
$$

---

Lasso adds an L1 penalty, which shrinks some coefficients to exactly zero.

This creates sparse models by effectively removing less important features.
Reducing the number of active features simplifies the model and improves generalization.


High
𝛼 α → More shrinkage → More coefficients become zero → Strong feature selection.

Low
𝛼 α → Behaves like standard linear regression → Higher risk of overfitting.

**Key Benefits of Lasso Regression**

 1. Reduces Variance → less overfitting.
 2. Feature Selection → sets some coefficients to zero.
 3. Improves Interpretability → simpler model with fewer predictors.

In [29]:
from sklearn.linear_model import Lasso

# Lasso Regression
lasso_reg = Lasso(alpha=0.01,max_iter=50000)
lasso_reg.fit(X_train, y_train)
y_pred_lasso = lasso_reg.predict(X_test)

print("Lasso Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))



Lasso Regression RMSE: 212539.51757695986


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.006e+13, tolerance: 2.259e+11



**ElasticNet**

Elastic Net Regression is a linear regression model that combines both L1 (Lasso) and L2 (Ridge) regularization. It is useful when you want the feature selection capability of Lasso and the stability of Ridge.

---

**Effect of Parameters**

1. alpha => controls overall regularization strength.
2. l1_ratio (ρ) => controls mix:
    - Closer to 1 : more Lasso effect (sparsity).
    - Closer to 0 : more Ridge effect (stability).

In [32]:
from sklearn.linear_model import ElasticNet

# Elastic Net Regression
elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic_net.fit(X_train, y_train)
y_pred_elastic = elastic_net.predict(X_test)

print("Elastic Net Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_elastic)))


Elastic Net Regression RMSE: 213354.04216794018
